# Clustering II: Hierarchical and Density-Based Clustering

In the previous tutorial, we covered K-Means and Gaussian Mixture Models (GMM). These methods are notoriously characterized by:

- Using Euclidean-based distances. For example, GMM uses $(x_i - \mu_C)^2$ in its probability distribution.
- Using a fixed number of clusters $k$.

In this tutorial, we will cover one widely used alternative algorithm that can go beyond these characterizations: **Hierarchical Clustering**. This technique allows you to inspect the structure of your data in a hierarchical way through the use of a tree of clusters. It also allows you to use any distance metric, i.e., you do not need to restrict yourself to Euclidean distances.

On the other hand, we did not mention it explicitly in the previous tutorial, but due to the way K-Means and GMM work, they tend to yield **spherical clusters**. As a result, they are not very good at dealing with datasets that have a more complex underlying structure. We will cover a widely known clustering technique that can deal with this situation: **Density-Based Clustering (DBSCAN)**.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
import matplotlib.pylab as plt

X, y = make_classification(n_samples=200, 
                           n_features=2, 
                           n_classes=4, 
                           n_informative=2, 
                           n_redundant=0, 
                           n_repeated=0, 
                           n_clusters_per_class=1,
                           random_state=1234,
                           class_sep=2,
                           #class_sep=1.2,
                          )

plt.scatter(X[:,0], X[:,1], c=y, cmap='Set2', s=100, linewidths=2, edgecolors="k")
plt.xlabel("Feature 1", size=18)
plt.ylabel("Feature 2", size=18)

## 1. Hierarchical Clustering

Hierarchical clustering builds a tree (called a **dendrogram**) showing how samples are merged step by step. 

There are two approaches:
- **Agglomerative (bottom-up):** each point starts as its own cluster, then merges occur.
- **Divisive (top-down):** start with one cluster and recursively split.

The *agglomerative* approach is the most common. In this approach, merging occurss given a **distance metric** and a **linkage criterion**.


This is the general procedure:

1. Each observation starts as an independent cluster.  
2. Choose a distance metric between observations (e.g., Euclidean, Cosine, Manhattan, etc.).  
3. Iteratively merge the two clusters with the smallest distance,  according to the selected linkage method.  
4. Continue merging until all observations belong to a single cluster.

### Linkage methods

The linkage method determines how “closeness” between clusters is defined, which directly affects the shape of the dendrogram and the resulting cluster structure.

1. **Single linkage**  
   The distance between two clusters is defined as the **minimum** distance between any two observations across the clusters.

   $$
   D(c_1, c_2) = \min_{i \in c_1,\, j \in c_2} d_{ij}
   $$

   It tends to produce long, chain-like clusters (sensitive to noise).


2. **Complete linkage**  
   The distance between two clusters is defined as the **maximum** distance between any two observations across the clusters.

   $$
   D(c_1, c_2) = \max_{i \in c_1,\, j \in c_2} d_{ij}
   $$

   It tends to yield compact and evenly sized clusters.


3. **Average linkage**  
   The distance between two clusters is the **average** pairwise distance between all observations in the two clusters.

   $$
   D(c_1, c_2) = 
   \frac{1}{N_{c_1} N_{c_2}} 
   \sum_{i \in c_1} \sum_{j \in c_2} d_{ij}
   $$

   It is a compromise between single and complete linkage.


4. **Ward linkage**  
   Instead of computing pairwise distances, Ward’s method merges the two clusters that result in the **smallest increase in total within-cluster variance**.  
   The distance between clusters is defined as:

   $$
   D(c_1, c_2) = 
   \frac{n_{c_1} n_{c_2}}{n_{c_1} + n_{c_2}} 
   \| \vec{\mu}_{c_1} - \vec{\mu}_{c_2} \|^2
   $$

   where:
   - $n_{c_1}, n_{c_2}$: number of observations in each cluster  
   - $\vec{\mu}_{c_1}, \vec{\mu}_{c_2}$: centroids of each cluster

   Due to $\| \vec{\mu}_{c_1} - \vec{\mu}_{c_2} \|^2$, it implicitly assumes Euclidean Distance.

   It tends to create clusters similar to K-Means.

### Implementation 1

There are several implementations in `Python`.

One can be found in `scipy.cluster.hierarchy`: 

- `linkage` performs hierarchical/agglomerative clustering.
- `dendrogram` plots the dendogram.
- `fcluster` allows to provide cluster labels for each observation.

This implementation is very useful when you need to **plot** your dendogram in an easy way.

In [ ]:
# CODE COMES HERE: Import the three functions just mentioned

In [ ]:
fig, axs = plt.subplots(figsize=(10, 7), ncols=2, nrows=2)
axs = axs.flatten()
for method, ax in zip(["single", "complete", "average", "ward"], axs):
    Z = None # CODE COMES HERE: Use linkage function with the input data X and the iterated linkage method
    # CODE COMES HERE: use dendogram function passing the tree Z, ax, setting no labels in the leaf and color_threshod=0
    ax.set_title(f"{method} linkage", size=20)
    ax.set_xlabel("Observations", size=18)
    ax.set_ylabel("Distance", size=18)

plt.tight_layout()

Each merge in the dendrogram represents the fusion of two clusters.

Cutting the dendrogram at a given **distance** (horizontal line) defines a clustering at that level. We can do this with `fcluster`.


In [ ]:
fig, axs = plt.subplots(figsize=(10, 7), ncols=2, nrows=2, sharex=True, sharey=True)
axs = axs.flatten()
for method, ax in zip(["single", "complete", "average", "ward"], axs):
    Z = None # CODE COMES HERE: Use linkage function with the input data X and the iterated linkage method
    clusters = None  # CODE COMES HERE: use fcluster passing the tree Z, t=4 and criterion='maxclust' for this to be the number of clusters
    ax.scatter(X[:,0], X[:,1], c=clusters, cmap='Set2', s=40, edgecolor='k')
    ax.set_title(f"Agglomerative Clustering \n (k=4; {method})", size=20)
    ax.set_xlabel("Feature 1", size=18)
    ax.set_ylabel("Feature 2", size=18)

plt.tight_layout()

### Implementation 2

Another implementation can be found in `AgglomerativeClustering`, which belongs to `cluster` module in `scikit-learn`.

Plotting is not as easy as in the `scipy` implementation.

In [ ]:
# CODE COMES HERE: Import AgglomerativeClustering

fig, axs = plt.subplots(figsize=(10, 7), ncols=2, nrows=2, sharex=True, sharey=True)
axs = axs.flatten()
for method, ax in zip(["single", "complete", "average", "ward"], axs):
    agg = None # CODE COMES HERE: Define AgglomerativeClustering object setting the number of clusters to 4 and the iterated linkage method
    # CODE COMES HERE: Fit agg to data
    clusters = None  # CODE COMES HERE: Retrive the labels from agg
    
    ax.scatter(X[:,0], X[:,1], c=clusters, cmap='Set2', s=40, edgecolor='k')
    ax.set_title(f"Agglomerative Clustering \n (k=4; {method})", size=20)
    ax.set_xlabel("Feature 1", size=18)
    ax.set_ylabel("Feature 2", size=18)

plt.tight_layout()

## 2. Density-Based Clustering (DBSCAN)

DBSCAN has the advantages of no needing to specify the number of cluster, finding clusters of arbitrary shapes and identifying noise points.

To motivate this new algorithm, let's  use the following data:

In [ ]:
from sklearn.datasets import make_blobs, make_moons
X, _ = make_moons(n_samples=300, noise=0.07, random_state=0)
X[np.argmax(X[:,1]),0] += 2.
X[np.argmin(X[:,0]),1] -= 1.

outlier_1 = X[np.argmax(X[:,1]),:].copy()
outlier_2 = X[np.argmin(X[:,0]),:].copy()

plt.figure(figsize=(5,4))
plt.scatter(X[:,0], X[:,1], edgecolor='k')
plt.xlabel("Feature 1", size=18)
plt.ylabel("Feature 2", size=18)
plt.show()

As you can see, there seems to be two clear clusters, although with a U-shape. 

Let's see what happens when we try to apply K-Means or GMM here looking for 2 clusters

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

fig, axs = plt.subplots(ncols=3, figsize=(15, 5))
km = None # CODE COMES HERE: Define K-means object with 2 clusters and a random_state
# CODE COMES HERE: Fit K-means object to data

axs[0].scatter(X[:,0], X[:,1], c=km.labels_, cmap="Set2", edgecolor='k')
axs[0].set_xlabel("Feature 1", size=18)
axs[0].set_ylabel("Feature 2", size=18)
axs[0].set_title("K-Means", size=20)

gmm = None # CODE COMES HERE: Define GaussianMixture object with 2 clusters and a random_state
# CODE COMES HERE: Fit GaussianMixture object to data

axs[1].scatter(X[:,0], X[:,1], c=gmm.predict(X), cmap="Set2", edgecolor='k')
axs[1].set_xlabel("Feature 1", size=18)
axs[1].set_ylabel("Feature 2", size=18)
axs[1].set_title("Gaussian Mixture Model", size=20)

agg = None # CODE COMES HERE: Define AgglomerativeClustering object with 2 clusters
# CODE COMES HERE: Fit AgglomerativeClustering object to data

axs[2].scatter(X[:,0], X[:,1], c=agg.labels_, cmap="Set2", edgecolor='k')
axs[2].set_xlabel("Feature 1", size=18)
axs[2].set_ylabel("Feature 2", size=18)
axs[2].set_title("Hierarchical Clustering", size=20)
plt.tight_layout()

They clearly fail to capture the real underlying structure of the data. 

Furthermore, those points we pushed away on purpose are also assigned to a given cluster, which as we suspect, they shouldn't.

In [ ]:
# Even GMM assigns these points to one cluster with high confidence
# CODE COMES HERE: Print probabilities for both outliers using GMM

Let's see that DBSCAN can really handle well these situations.

This algorithm can be found in the `cluster` module with the name `DBSCAN`


In [ ]:
# CODE COMES HERE:Import DBSCAN

db = None # CODE COMES HERE: create DBSCAN object eps=0.3, min_samples=5
# CODE COMES HERE: Fit DBSCAN to data
labels = None # CODE COMES HERE: retrieve labels attribute
plt.figure(figsize=(7,4))
plt.scatter(X[:,0], X[:,1], c=labels, cmap='Set2', edgecolor='k')
plt.title(f"DBSCAN", size=20)
plt.xlabel("Feature 1", size=18)
plt.ylabel("Feature 2", size=18)
plt.colorbar()
plt.show()

Nice!!! This is how DBSCAN works: 

- It considers clusters as **dense regions** of points separated by regions of **low density**.
- Density is defined by a number of minimal samples within the clustr and a radius.
- Given the density and radius, DBSCAN works with the following type of points:  
      - Core observations, which are points that are within the defined radius and in a dense region. Any core observation is part of a cluster, by definition.  
      - Border observations, which are points that interact with other observations within the defined radius but are not dense enough.  
      - Noise observations, which are points whose radius do not include any other observations. (The green points above)  

In [ ]:
# Choose a point to highlight (example: highest y-value)
idx = 230
pt = X[idx]

# ---- Plot ----
plt.figure(figsize=(5,4))
plt.scatter(X[:,0], X[:,1], edgecolor='k')
plt.scatter(pt[0], pt[1], c="red", edgecolor='k')

# Draw a circle around that point
circle = plt.Circle((pt[0], pt[1]), radius=.2, 
                    fill=False, color='red', linewidth=2)
plt.gca().add_patch(circle)

plt.xlabel("Feature 1", size=18)
plt.ylabel("Feature 2", size=18)

plt.show()


Key parameters:
- `eps`: radius of the neighborhood around a point
- `min_samples`: minimum number of points required to form a dense region

Let's see how results change according to these parameters:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(10,10), sharex=True, sharey=True)
axs = axs.flatten()
for eps, ax in zip([0.1, 0.2, 0.3, 0.5], axs):
    db = None # CODE COMES HERE: create DBSCAN object min_samples=5 and the iterated eps
    # CODE COMES HERE: Fit DBSCAN to data
    labels = None # CODE COMES HERE: retrieve labels attribute
    ax.scatter(X[:,0], X[:,1], c=labels, cmap='Set2', s=40, edgecolor='k')
    ax.set_title(f"eps = {eps}", size=20)
    ax.set_xlabel("Feature 1", size=18)
    ax.set_ylabel("Feature 2", size=18)
    
plt.tight_layout()

Notice how smaller `eps` values create more clusters and isolate points, while larger `eps` values merge clusters together.

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(10,10), sharex=True, sharey=True)
axs = axs.flatten()
for min_samples, ax in zip([1, 5, 10, 20], axs):
    db = None # CODE COMES HERE: create DBSCAN object with eps=0.2 and itereated min_samples
    # CODE COMES HERE: Fit DBSCAN to data
    labels = None # CODE COMES HERE: retrieve labels attribute
    
    ax.scatter(X[:,0], X[:,1], c=labels, cmap='Set2', s=40, edgecolor='k')
    ax.set_title(f"min_samples = {min_samples}", size=20)
    ax.set_xlabel("Feature 1", size=18)
    ax.set_ylabel("Feature 2", size=18)
    
plt.tight_layout()

Points labeled as `-1` are considered **noise**.

## 📝 3. Exercises

### 3.1 Clustering the Iris Dataset (again)

Try DBSCAN and Agglomerative on the Iris dataset.
- Standardize the features.
- Visualize the data by applying PCA and color each point according to their true labels.
- Test DBSCAN with different `eps` and  `min_samples` values. Visualize your results reducting your data to 2 dimensions after applying PCA.
- Experiment with hierarchical clustering using the different linkages options: "single", "complete", "average", "ward". For each linkage, try to determine the right number of clusters using the silhouette score. Plot each solution and take that one In particular, as we did with our data, show how results chan
  

In [ ]:
from sklearn.datasets import load_iris
X, y = load_iris()["data"], load_iris()["target"]

In [ ]:
# YOUR CODE

### 3.2 Hierarchical clustering on Digits data

Your goal here is to try to do something similar to the example provided by scikit-learn: https://scikit-learn.org/stable/auto_examples/cluster/plot_digits_linkage.html

Specifically:

- Apply a PCA to reduce the dimensionality of the data to 2 dimensions.
- Apply hierarchical clustering to this reduced data searching for 10 clusters and trying the four different linkages ("ward", "average", "complete", "single").
- For each linkage case, plot the distribution of points, coloring observations according to the labels assigned by hierarchical clustering.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
n_samples, n_features = X.shape

In [ ]:
# YOUR CODE